### Explanation 1
This cell prints the Python interpreter version. In a viva, you can say it is used to verify the runtime environment before running TensorFlow or other dependent libraries.


In [ ]:
!python --version

Python 3.12.12


### Explanation 2
This cell imports the libraries required for the experiment, such as TensorFlow, Keras, layers, and dataset helpers. It is the initialization step that makes the later model-building and training code possible.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential,layers
import tensorflow_datasets as tfds

### Explanation 3
This cell loads the IMDB reviews dataset from TensorFlow Datasets. It returns both the dataset splits and metadata, which are later used for text preprocessing and binary sentiment classification.


In [ ]:
(ds_train,ds_test),ds_info=tfds.load(
    'imdb_reviews',
    split=['train','test'],
    as_supervised=True,
    with_info=True
)

### Explanation 4
This cell displays a few raw dataset samples. It is a sanity-check step used to inspect the text-label pairs before applying preprocessing or vectorization.


In [ ]:
i=0
for text,label in ds_train:
  print(text,label)
  i=i+1
  if (i==5):
    break;

tf.Tensor(b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.", shape=(), dtype=string) tf.Tensor(0, shape=(), dtype=int64)
tf.Tensor(b'I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on t

### Explanation 5
This cell downloads the NLTK corpora and tokenizers required for text preprocessing. Without these resources, operations such as tokenization, stopword filtering, and lemmatization would fail.


In [ ]:
import nltk
from nltk.corpus import stopwords
import re
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
stopwords_list=stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


### Explanation 6
This cell defines the text preprocessing pipeline. It converts text to lowercase, removes unwanted characters, tokenizes the sentence, removes stopwords, and may apply lemmatization so that semantically similar word forms are normalized.


In [ ]:
def preprocessing(text,label):
  text = text.numpy().decode('utf-8')
  text=text.lower()
  text=re.sub(r'[^a-z\s]','',text)
  tokenized_text=nltk.word_tokenize(text)
  preprocessed_text=' '.join([WordNetLemmatizer().lemmatize(word) for word in tokenized_text if word not in stopwords_list])
  return preprocessed_text,label


### Explanation 7
This cell wraps the Python preprocessing function using `tf.py_function` so it can be inserted into a `tf.data` pipeline. The explicit shape setting is important because TensorFlow cannot infer shapes automatically from arbitrary Python code.


In [ ]:
def preprocessing_py(text,label):
  text,label=tf.py_function(
      preprocessing,
      inp=[text,label],
      Tout=[tf.string,label.dtype]
  )
  text.set_shape([])
  label.set_shape([])
  return text,label

### Explanation 8
This cell applies the preprocessing function to each element of the dataset through the `map` transformation. After this step, the raw text stream is converted into cleaned text samples.


In [ ]:
ds_train=ds_train.map(preprocessing_py)

### Explanation 9
This cell applies the preprocessing function to each element of the dataset through the `map` transformation. After this step, the raw text stream is converted into cleaned text samples.


In [ ]:
ds_test=ds_test.map(preprocessing_py)

### Explanation 10
This cell imports the libraries required for the experiment, such as TensorFlow, Keras, layers, and dataset helpers. It is the initialization step that makes the later model-building and training code possible.


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer=Tokenizer(num_words=10000,oov_token='UNK')

### Explanation 11
This cell extracts the training text into a Python list of strings. That list is then used to build the tokenizer vocabulary from the training corpus.


In [ ]:
train_texts=[text.numpy().decode('utf-8') for text,_ in ds_train]

### Explanation 12
This cell fits the tokenizer on the training text. Technically, it builds a word-index dictionary based on token frequency so that text can be converted into numeric sequences.


In [ ]:
tokenizer.fit_on_texts(train_texts)

### Explanation 13
This cell defines a function that converts cleaned text into integer token sequences and then pads them to a fixed length. Fixed-length sequences are required because dense batches must have uniform shape.


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def encoding(text, label):
    # Tensor → Python
    text = text.numpy().decode("utf-8")

    # Tokenizer expects a LIST
    seq = tokenizer.texts_to_sequences([text])

    # Pad
    seq = pad_sequences(seq, maxlen=256)

    return seq[0], label


### Explanation 14
This cell wraps the encoding function with `tf.py_function` so it can run inside the TensorFlow input pipeline. It also fixes the static output shape, which is necessary for batching and model input compatibility.


In [ ]:
def encoding_py(text, label):
    seq, label = tf.py_function(
        func=encoding,
        inp=[text, label],          # ✅ REAL tensors
        Tout=[tf.int32, tf.int64]
    )
    seq.set_shape([None])
    label.set_shape([])
    return seq, label


### Explanation 15
This cell applies the numeric encoding step to each dataset example. After this transformation, the dataset contains tensors instead of raw strings, so it is ready for model training.


In [ ]:
ds_train = ds_train.map(encoding_py)
ds_test  = ds_test.map(encoding_py)


### Explanation 16
This cell groups samples into mini-batches. Mini-batch training is computationally efficient and produces gradient estimates that are more stable than single-sample updates.


In [ ]:
ds_train=ds_train.batch(64)
ds_test=ds_test.batch(64)

### Explanation 17
This cell loads the IMDB reviews dataset from TensorFlow Datasets. It returns both the dataset splits and metadata, which are later used for text preprocessing and binary sentiment classification.


In [ ]:
(ds_train,ds_test),ds_info=tfds.load(
    'imdb_reviews',
    split=['train','test'],
    as_supervised=True,
    with_info=True
)
vectorizer = layers.TextVectorization(
    max_tokens=10000,
    output_sequence_length=256
)

vectorizer.adapt(ds_train.map(lambda x, y: x))

ds_train = ds_train.map(lambda x, y: (vectorizer(x), y))
ds_test  = ds_test.map(lambda x, y: (vectorizer(x), y))


### Explanation 18
This cell batches the dataset and enables prefetching. Prefetch overlaps input preparation with model execution, which improves pipeline throughput during training.


In [ ]:
BATCH_SIZE = 32

ds_train = ds_train.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test  = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


### Explanation 19
This cell defines a text classification model with an embedding layer followed by pooling and dense layers. The embedding layer learns dense vector representations of words, which the classifier then uses for sentiment prediction.


In [ ]:
model=keras.Sequential([
    keras.layers.Embedding(10000,512),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(64,activation='relu'),
    keras.layers.Dense(1,activation='sigmoid')

])

### Explanation 20
This cell executes an intermediate step in the machine learning workflow. Depending on context, it prepares data, defines model behavior, or runs part of training or evaluation.


In [ ]:
optimizer=keras.optimizers.Adam()
loss_function=keras.losses.BinaryCrossentropy()
acc_metric=keras.metrics.BinaryAccuracy()

### Explanation 21
This cell implements a custom training loop with `tf.GradientTape`. Instead of relying on `model.fit`, it manually performs the forward pass, computes loss, calculates gradients, and updates model weights.


In [ ]:
epochs=5
for i in range(epochs):
  print(f'\n Training started for epoch {i}')
  for batch_id,(x_batch,y_batch) in enumerate(ds_train):
    with tf.GradientTape() as tape:
      y_pred=model(x_batch,training=True)
      loss=loss_function(y_batch,y_pred)
    gradients=tape.gradient(loss,model.trainable_weights)
    optimizer.apply_gradients(zip(gradients,model.trainable_weights))
    acc_metric.update_state(y_batch,y_pred)
  accuracy=acc_metric.result()
  print(f'Accuracy over epoch {accuracy}')
  acc_metric.reset_state()



 Training started for epoch 0
Accuracy over epoch 0.7859600186347961

 Training started for epoch 1
Accuracy over epoch 0.8847600221633911

 Training started for epoch 2
Accuracy over epoch 0.9087600111961365

 Training started for epoch 3
Accuracy over epoch 0.9192799925804138

 Training started for epoch 4
Accuracy over epoch 0.924239993095398


### Explanation 22
This cell evaluates the model on the test dataset batch by batch. The metric object accumulates accuracy across batches and then reports the final test performance.


In [ ]:
for batch_index,(x_batch,y_batch) in enumerate(ds_test):
  y_pred=model(x_batch,training=False)
  acc_metric.update_state(y_batch,y_pred)
test_accuracy=acc_metric.result()
print(f'Test Accuracy {test_accuracy}')

Test Accuracy 0.8025599718093872


### Explanation 23
This is an empty placeholder cell. It does not execute anything, but it can be used later to add another step in the workflow.
